In [2]:
import torch
import transformers

print(torch.__version__)
print(transformers.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

2.10.0+cu128
5.8.1
True
NVIDIA RTX A5000


In [3]:
import pandas as pd

df = pd.read_csv(
    "../data/processed_data&code/amp_nonamp_dataset.csv"
)

print(df.shape)
df.head()

(16800, 2)


,Sequence,label
0,ISGINASVVNIQKEIDRLNEVAKNLNESLIK,1
1,WAIVLL,1
2,INVLGILGLLGKALSHL,1
3,FPPWVL,1
4,AVTSSVADTTTVVRDDF,0


In [4]:
test_df = df.sample(
    100,
    random_state=42
).reset_index(drop=True)

print(test_df.shape)

(100, 2)


In [5]:
import torch
import esm

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model, alphabet = esm.pretrained.esm2_t33_650M_UR50D()

model = model.to(device)
model.eval()

batch_converter = alphabet.get_batch_converter()

print("Loaded")

Loaded


In [6]:
import torch
import numpy as np
from tqdm import tqdm

embeddings = []

for seq in tqdm(test_df["Sequence"]):

    batch = [("protein", seq)]

    _, _, tokens = batch_converter(batch)

    tokens = tokens.to(device)

    with torch.no_grad():

        result = model(
            tokens,
            repr_layers=[33],
            return_contacts=False
        )

    token_embeddings = result[
        "representations"
    ][33]

    seq_embedding = (
        token_embeddings[:, 1:-1]
        .mean(1)
        .squeeze()
        .cpu()
        .numpy()
    )

    embeddings.append(seq_embedding)

embeddings = np.array(embeddings)

print(embeddings.shape)

100%|█████████████████████████████████████████| 100/100 [00:01<00:00, 64.32it/s]

(100, 1280)


In [7]:
np.save(
    "test_embeddings.npy",
    embeddings
)

In [8]:
import numpy as np
from tqdm import tqdm

sequences = df["Sequence"].tolist()

batch_size = 64

all_embeddings = []

for i in tqdm(range(0, len(sequences), batch_size)):

    batch_seqs = sequences[i:i+batch_size]

    batch = [
        (str(j), seq)
        for j, seq in enumerate(batch_seqs)
    ]

    _, _, tokens = batch_converter(batch)

    tokens = tokens.to(device)

    with torch.no_grad():

        results = model(
            tokens,
            repr_layers=[33],
            return_contacts=False
        )

    reps = results["representations"][33]

    for j, seq in enumerate(batch_seqs):

        seq_len = len(seq)

        emb = (
            reps[j, 1:seq_len+1]
            .mean(0)
            .cpu()
            .numpy()
        )

        all_embeddings.append(emb)

embeddings = np.array(all_embeddings)

print(embeddings.shape)

100%|█████████████████████████████████████████| 263/263 [02:18<00:00,  1.90it/s]

(16800, 1280)


In [9]:
np.save(
    "amp_nonamp_embeddings.npy",
    embeddings
)

df.to_csv(
    "amp_nonamp_metadata.csv",
    index=False
)

print("Saved")

Saved
